# 03 - Model Evaluation
Loads the saved artifacts (exactly what the API serves) and inspects test-set performance and the clinical threshold.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.data.preprocess import read_patient_csv  # keeps the 'None' alcohol category intact
from src.features.schema import NUMERIC_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES, FEATURE_COLUMNS, TARGET
import json, joblib
from src.models.evaluate import compute_metrics
from src.models.risk import risk_level

In [ ]:
model = joblib.load('models/readmission_model.joblib')
threshold = json.load(open('models/threshold.json'))
metadata = json.load(open('models/model_metadata.json'))
print(threshold)
metadata['model_name'], metadata['metrics']

In [ ]:
pd.read_csv('reports/evaluation/model_comparison.csv')

In [ ]:
test = read_patient_csv('data/processed/test.csv')
proba = model.predict_proba(test[FEATURE_COLUMNS])[:, 1]
compute_metrics(test[TARGET], proba, threshold['threshold'])

## Threshold trade-off
Lower threshold -> higher recall (fewer missed readmissions) but more false alarms. We require recall >= 0.75.

In [ ]:
rows = [compute_metrics(test[TARGET], proba, t) for t in np.arange(0.05, 0.65, 0.05)]
pd.DataFrame(rows)[['threshold', 'precision', 'recall', 'f1', 'accuracy']]

In [ ]:
pd.Series([risk_level(p) for p in proba]).value_counts()

In [ ]:
from IPython.display import Image
for f in ['confusion_matrix', 'pr_curve', 'calibration_curve', 'shap_summary']:
    display(Image(f'reports/figures/{f}.png'))